# FloodNet — treatment

Turns the raw FloodNet export into two files the router can use.

**The problem this notebook solves:** the export has no latitude or longitude
column. Location exists only inside `Sensor Name`, as a borough prefix plus a
street — `Q - Beach 84 St`, `BX - Ditmars St/Hunter Ave 2`. Every sensor has to
be parsed out and geocoded once before the flood layer can score anything.

**Outputs**

| File | Rows | What |
|---|---|---|
| `floodnet_events_clean.csv` | ~2,929 | One row per flood event, typed and parsed, time-series traces dropped |
| `flood_sensors.csv` | ~294 | One row per sensor with coordinates, event count and depth stats — this is what the router reads |

Geocoding uses **Nominatim** (OpenStreetMap): no API key, and ODbL permits
storing the results, which Mapbox's standard endpoint does not. It is rate
limited to one request per second, so the full pass takes about six minutes.
Results are cached to `geocode_cache.json`, so re-running is instant and an
interrupted run resumes where it stopped.

In [ ]:
from __future__ import annotations

import json
import re
import time
from pathlib import Path

import pandas as pd
import requests

# Raw datasets live in data/ at the repo root; this notebook lives in notebooks/.
DATA_DIR = Path("../data")
FILENAME = "FloodNet__Street_Flooding_Events_Measured_by_FloodNet_Sensors_20260815.csv"

DATA_PATH = next((p for p in [DATA_DIR / FILENAME, Path(FILENAME)] if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"{FILENAME} not found in {DATA_DIR.resolve()}")

OUT_DIR = DATA_DIR
CACHE_PATH = OUT_DIR / "geocode_cache.json"

print(f"Reading {DATA_PATH.resolve()}")

## 1 · Load

The two `Time Series ...` columns carry the full depth trace for each event —
up to ~45,000 characters per row. Nothing downstream uses them, and they
dominate both read time and memory, so they are skipped at parse time rather
than dropped afterwards.

In [ ]:
raw = pd.read_csv(DATA_PATH, usecols=lambda c: not c.startswith("Time Series"))

print(f"{len(raw):,} events x {raw.shape[1]} columns")
raw.head(3)

## 2 · Types

Source dates are `MM/DD/YYYY HH:MM:SS AM/PM`. Parsing with an explicit format
rather than letting pandas infer means a change in the export shows up as NaT
here instead of as silently wrong dates later.

In [ ]:
COLUMNS = {
    "Sensor Name": "sensor_name",
    "Sensor ID": "sensor_id",
    "Flood Start Datetime (GMT)": "start_gmt",
    "Flood End Datetime (GMT)": "end_gmt",
    "Maximum Flood Depth (inches)": "max_depth_in",
    "Time to Maximum Flood Depth (minutes)": "mins_to_peak",
    "Time to Drain From Peak (minutes)": "mins_to_drain",
    "Total Duration (minutes)": "duration_min",
    "Duration of Flooding Greater Than 4 Inches (minutes)": "mins_over_4in",
    "Duration of Flooding Greater Than 12 Inches (minutes)": "mins_over_12in",
    "Duration of Flooding Greater Than 24 Inches (minutes)": "mins_over_24in",
}
df = raw.rename(columns=COLUMNS)

for col in ["start_gmt", "end_gmt"]:
    df[col] = pd.to_datetime(df[col], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

NUMERIC = [
    "max_depth_in", "mins_to_peak", "mins_to_drain", "duration_min",
    "mins_over_4in", "mins_over_12in", "mins_over_24in",
]
df[NUMERIC] = df[NUMERIC].apply(pd.to_numeric, errors="coerce")

print("unparsed start times:", int(df["start_gmt"].isna().sum()))
print("date range:", df["start_gmt"].min(), "->", df["start_gmt"].max())
df[NUMERIC].describe().T

## 3 · Pull location out of the sensor name

`Q - Beach 84 St` splits into a borough prefix and a street. Three traps:

- **The separator is not consistent.** Most names use `" - "`, but at least one
  (`SI- RICHMOND TER/LAFAYETTE AVE`) has no leading space. Splitting on `" - "`
  silently swallows the whole string as the prefix.
- **You cannot just split on the first `-` instead.** Queens uses hyphenated
  house numbers (`150-15 Archer Ave`), so that would cut streets in half. The
  regex below anchors on the known borough codes, which sidesteps both.
- **Some sensors are disambiguated by a trailing marker** — `Ditmars St/Hunter Ave 2`
  and `Beach 84 St (2)` are extra units at a corner, not house numbers. Both forms
  are stripped. Only *trailing* markers go, which leaves street numbers like
  `Beach 84 St` intact.

The parse is asserted, not assumed. A new borough code or a new name format
fails loudly here rather than silently dropping sensors.

In [ ]:
BOROUGHS = {
    "M": "Manhattan", "MN": "Manhattan",
    "BX": "Bronx",
    "BK": "Brooklyn", "BKN": "Brooklyn",
    "Q": "Queens", "QN": "Queens",
    "SI": "Staten Island",
}

# Longer codes first so MN wins over M, BKN over BK, QN over Q.
PREFIX_RE = re.compile(r"^(MN|BKN|BX|BK|QN|SI|M|Q)\s*-\s*(.+)$", re.IGNORECASE)


def split_sensor_name(name: str) -> tuple[str | None, str | None]:
    match = PREFIX_RE.match(str(name).strip())
    if not match:
        return None, None
    return match.group(1).upper(), match.group(2).strip()


parsed = [split_sensor_name(n) for n in df["sensor_name"]]
df["borough_code"] = [p[0] for p in parsed]
df["location_raw"] = [p[1] for p in parsed]

unparsed = df.loc[df["borough_code"].isna(), "sensor_name"].unique()
assert len(unparsed) == 0, f"Could not parse {len(unparsed)} sensor names: {unparsed[:5]}"

df["borough"] = df["borough_code"].map(BOROUGHS)


def clean_location(raw: str) -> str:
    s = str(raw).strip()
    s = re.sub(r"\s*\(\d+\)\s*$", "", s)   # "Beach 84 St (2)"
    s = re.sub(r"\s+\d+$", "", s)          # "Ditmars St/Hunter Ave 2"
    s = re.sub(r"\s*/\s*", "/", s)         # "Brookville Blvd/ Snake Rd"
    return re.sub(r"\s+", " ", s).strip()


df["location"] = df["location_raw"].map(clean_location)

print(df["borough"].value_counts().to_string())
print(f"\n{df['sensor_name'].nunique()} sensors at {df['location'].nunique()} distinct corners")
df[["sensor_name", "borough", "location"]].drop_duplicates().head(8)

## 4 · Sanity check — is flooding accelerating?

Worth confirming before building on it, and it is the strongest single fact this
dataset gives you.

In [ ]:
by_year = df.assign(year=df["start_gmt"].dt.year).groupby("year").size()
print(by_year.to_string())

print(f"\nmedian peak depth: {df['max_depth_in'].median():.2f} in")
print(f"events reaching 4+ in: {int((df['max_depth_in'] >= 4).sum()):,}")

## 5 · Collapse to corners, not hardware

The router needs *places to avoid*, and a place is a street corner, not a piece
of equipment. Several corners carry two or three sensors — `Beach 84 St` and
`Beach 84 St (2)` are the same intersection — and leaving them separate would
count the same flood twice when scoring nearby road segments.

So two tables: one per sensor for provenance, and one per corner, which is what
everything downstream actually reads.

In [ ]:
AGGS = dict(
    event_count=("sensor_name", "size"),
    max_depth_in=("max_depth_in", "max"),
    median_depth_in=("max_depth_in", "median"),
    events_over_4in=("max_depth_in", lambda s: int((s >= 4).sum())),
    first_event=("start_gmt", "min"),
    last_event=("start_gmt", "max"),
)

sensors = (
    df.groupby(["sensor_name", "borough", "location"], as_index=False)
    .agg(**AGGS)
    .sort_values("event_count", ascending=False)
    .reset_index(drop=True)
)

corners = (
    df.groupby(["borough", "location"], as_index=False)
    .agg(sensor_count=("sensor_name", "nunique"), **AGGS)
    .sort_values("event_count", ascending=False)
    .reset_index(drop=True)
)

print(f"{len(sensors)} sensors -> {len(corners)} corners")
corners.head(10)

## 6 · Geocode

Three things make this work, and the first one is not optional:

**Numbered streets need ordinal suffixes.** FloodNet writes `Beach 84 St`; OSM
stores it as `Beach 84th Street`, and the bare form matches *nothing*. That one
sensor is 438 events — 15% of the entire dataset, the worst location in New York
— so without this step the flood layer loses its single most important point and
still looks like it worked.

**Intersections degrade to one street.** Nominatim has no real intersection
search, so `A/B` is tried as `A & B` and then falls back to the first street
alone. Fine at our 200 m matching radius — a corner and one of its streets are
the same place.

**`bounded=1` with a NYC viewbox** stops the geocoder wandering off to a
same-named street in another state, the usual failure mode for generic names.

**~6 minutes on a cold cache.** Progress saves every 25 lookups, so an
interrupted run loses at most 25.

In [ ]:
NOMINATIM = "https://nominatim.openstreetmap.org/search"
# Nominatim's usage policy requires a real identifying User-Agent.
HEADERS = {"User-Agent": "RiskMap-hackathon/0.1 (matiaspfreire@gmail.com)"}
NYC_VIEWBOX = "-74.30,40.93,-73.68,40.47"  # W,N,E,S

cache: dict = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
print(f"{len(cache)} lookups already cached")


STREET_TYPE = r"(?:St|Street|Ave|Avenue|Rd|Road|Dr|Drive|Blvd|Pl|Place|Ter|Terrace|Ln|Lane|Ct|Court|Way)"


def _ordinal_suffix(n: int) -> str:
    if 11 <= n % 100 <= 13:
        return "th"
    return {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")


def ordinalize(text: str) -> str:
    """'Beach 84 St' -> 'Beach 84th St'.

    Only a bare number directly before a street type is touched, so Queens
    hyphenated house numbers like '150-15 Archer Ave' are left alone.
    """
    def repl(match):
        n = int(match.group(1))
        return f"{n}{_ordinal_suffix(n)} {match.group(2)}"

    return re.sub(rf"\b(\d+)\s+({STREET_TYPE})\b", repl, text)


def queries_for(location: str, borough: str) -> list[str]:
    """Progressively looser queries, most specific first."""
    out = []
    for text in dict.fromkeys([ordinalize(location), location]):
        for candidate in dict.fromkeys([text, text.split("/")[0].strip()]):
            query = f"{candidate.replace('/', ' & ')}, {borough}, New York, NY"
            if query not in out:
                out.append(query)
    return out


def geocode(location: str, borough: str) -> dict | None:
    key = f"{borough}|{location}"
    if key in cache:
        return cache[key]

    result = None
    for q in queries_for(location, borough):
        try:
            r = requests.get(
                NOMINATIM,
                params={
                    "q": q, "format": "json", "limit": 1,
                    "countrycodes": "us", "viewbox": NYC_VIEWBOX, "bounded": 1,
                },
                headers=HEADERS,
                timeout=20,
            )
            r.raise_for_status()
            hits = r.json()
        except Exception as exc:
            print(f"  ! {q}: {exc}")
            hits = []

        time.sleep(1.1)  # one request per second, per Nominatim's policy

        if hits:
            result = {"lat": float(hits[0]["lat"]), "lon": float(hits[0]["lon"]), "query": q}
            break

    cache[key] = result
    return result

In [ ]:
for i, row in corners.iterrows():
    geocode(row["location"], row["borough"])
    if (i + 1) % 25 == 0:
        CACHE_PATH.write_text(json.dumps(cache))
        print(f"  {i + 1}/{len(corners)}")

CACHE_PATH.write_text(json.dumps(cache))

hits = [cache.get(f"{b}|{l}") for b, l in zip(corners["borough"], corners["location"])]
corners["lat"] = [h["lat"] if h else None for h in hits]
corners["lon"] = [h["lon"] if h else None for h in hits]

found = int(corners["lat"].notna().sum())
print(f"\ngeocoded {found}/{len(corners)} corners ({found / len(corners):.1%})")

## 7 · Validate

Coordinates must land inside NYC, and what failed has to be judged by **events
lost, not corners lost**. Flooding is extremely concentrated, so missing 30 rural
corners with one event each is nothing, while missing `Beach 84 St` alone would
cost 15% of the dataset. Rows are sorted by event count — a failure near the top
is worth fixing by hand, one at the bottom is not.

In [ ]:
W, S, E, N = -74.30, 40.47, -73.68, 40.93
located = corners["lat"].notna()
in_box = corners["lon"].between(W, E) & corners["lat"].between(S, N)
good = located & in_box

print("outside the NYC bounding box:", int((located & ~in_box).sum()))
print("no coordinates at all:", int((~located).sum()))

missed = corners.loc[~good, ["borough", "location", "event_count"]]
lost = int(missed["event_count"].sum())
print(f"\nevents kept:  {int(corners.loc[good, 'event_count'].sum()):,} "
      f"({corners.loc[good, 'event_count'].sum() / len(df):.1%})")
print(f"events lost:  {lost:,} ({lost / len(df):.1%}) across {len(missed)} corners")
missed.head(20)

## 8 · Write

`flood_sensors.csv` is the file the router consumes — one row per corner, with
coordinates and an event count to scale the flood penalty by.

In [ ]:
sensors_out = OUT_DIR / "flood_sensors.csv"
events_out = OUT_DIR / "floodnet_events_clean.csv"

corners.loc[good].to_csv(sensors_out, index=False)
df.drop(columns=["borough_code", "location_raw"]).to_csv(events_out, index=False)

print(f"{sensors_out.name}          -> {int(good.sum())} corners")
print(f"{events_out.name} -> {len(df):,} events")

print("\nWorst corners by event count:")
print(
    corners.loc[good, ["borough", "location", "sensor_count", "event_count", "max_depth_in", "lat", "lon"]]
    .head(10)
    .to_string(index=False)
)